In [1]:
import numpy as np
import itertools
import pennylane as qml
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

## Optimizer

In [2]:
class Optimizer():
    """
    Base class for all optimizers
    """
    def update(self, w: dict, grad: dict) -> dict:
        pass


class SGD(Optimizer):
    def __init__(self, lr: float):
        """
        SGD optimizer

        Args:
            lr: learning rate
        """
        self.lr = lr

    def update(self, w: dict, grad: dict) -> dict:
        """
        Update weight

        Args:
            w: weight
            grad: gradient
        """
        for key, value in w.items():
            w[key] -= self.lr * grad[key]


        return w
    
class Momentum(Optimizer):
    def __init__(self, lr: float, gamma: float = 0.9):
        """
        Momentum optimizer

        Args:
            lr: learning rate
            gamma: momentum coefficient
        """
        self.lr = lr
        self.gamma = gamma
        self.v = None  # Velocity

    def update(self, w: dict, grad: dict) -> dict:
        """
        Update weight

        Args:
            w: weight
            grad: gradient
        """

        # If this is the first iteration, initialize v with zeros with same shape as w
        if self.v is None:
            self.v = {
                key: np.zeros_like(value) for key, value in w.items()
            }

        for key, value in self.v.items():
            self.v[key] += self.lr * grad[key]
            w[key] -= self.v[key] 

        return w

## Circuits

In [3]:
class BrickWallCircuit():
    def __init__(self, n_qubits, num_layers, num_RZZ=None, gate_type = None, para=None):


        self.get_key_local = lambda i,j : "{}-layer {}-index local".format(i, j)
        self.get_key_CZ = lambda i,j : "{}-layer {}-index CZ".format(i, j)
        self.get_key_gate_type = lambda i,j : "{}-layer {}-index gate type".format(i,j)

        self.n_qubits = n_qubits
        self.num_layers = num_layers

        if gate_type is not None:
            self.gate_type = gate_type
        elif gate_type is None and num_RZZ is not None:
            self.generate_gate_type(num_RZZ)
        else:
            self.generate_gate_type(2)
        if para is None:
            self.generate_parameters()
        else:
            self.para = para

    def apply_circuit(self, wires=None):
        """
        Apply the circuit operations to the given wires
        This is used inside a PennyLane QNode
        """
        if wires is None:
            wires = range(self.n_qubits)
        
        for layer in range(self.num_layers):
            self._Layer(layer, wires)

    def _Layer(self, layer_index, wires):

        index = [2*i for i in range((self.n_qubits // 2))] if layer_index % 2 == 0 else [2*i+1 for i in range((self.n_qubits-1) // 2 )]

        for i in range(len(index)):
            local_para = self.para[self.get_key_local(layer_index, index[i])]
            Rzz_para = self.para[self.get_key_CZ(layer_index, index[i])]
            num_RZZ = self.gate_type[self.get_key_gate_type(layer_index, index[i])]
            
            self._bricks(local_para, num_RZZ, Rzz_para, index[i], wires)
    
    def _bricks(self, local_para, num_RZZ, Rzz_para, index, wires):
        q0_para = local_para[0, :]
        q1_para = local_para[1, :]

        self._local(q0_para, wires[index])
        self._local(q1_para, wires[index+1])

        for i in range(num_RZZ):
            q0_para = local_para[2*i+2, :]
            q1_para = local_para[2*i+3, :]

            qml.IsingZZ(Rzz_para[i], wires=[wires[index], wires[index+1]])
            self._local(q0_para, wires[index])
            self._local(q1_para, wires[index+1])

    def _local(self, para, wire):
        qml.RZ(para[0], wires=wire)
        qml.RX(para[1], wires=wire)
        qml.RZ(para[2], wires=wire)

    def generate_parameters(self):
        """
        This function generates random parameters for the circuit automatically.
        Given gate_type, it generates parameters for the gate of the circuit.
        The parameters are stored in self.para which is a dictionary.
        """
        self.para = {}
        for i in range(self.num_layers):
            if i % 2 == 0:
                num_index = self.n_qubits//2
                index = [2*j for j in range(num_index)]
            else:
                num_index = (self.n_qubits-1)//2
                index = [2*j+1 for j in range(num_index)]
    

            for k in range(num_index):
                temp_para = np.random.random((2*self.gate_type[self.get_key_gate_type(i,index[k])]+2, 3)) * np.pi * 2
                temp_Rzz_para = np.random.random(self.gate_type[self.get_key_gate_type(i,index[k])]) * np.pi * 2 if self.gate_type[self.get_key_gate_type(i,index[k])] > 0 else None
    
                self.para[self.get_key_local(i,index[k])] = temp_para
                self.para[self.get_key_CZ(i,index[k])] = temp_Rzz_para

    def generate_gate_type(self, num_RZZ):
        """
        This function generates gate_type for the circuit automatically.
        Given the number of CZ gates, it generates parameters for the gate of the circuit.
        The parameters are stored in self.para which is a dictionary.
        """
        self.gate_type = {}
        for i in range(self.num_layers):
            if i % 2 == 0:
                num_index = self.n_qubits//2
                index = [2*j for j in range(num_index)]
            else:
                num_index = (self.n_qubits-1)//2
                index = [2*j+1 for j in range(num_index)]
    
            for k in range(num_index):
                self.gate_type[self.get_key_gate_type(i, index[k])] = num_RZZ

In [4]:
class EntanglementDetection:
    def __init__(self, target_state_prep_fn, 
                test_state_circuit_config, 
                optimizer, 
                n_qubits=2, 
                mode='partial_transpose', 
                use_parallel=False,
                shots=10000,
                ):
        """
        Initialize VED for entanglement detection
        
        Args:
            target_state_prep_fn: Function that prepares the target state
            n_qubits: Number of qubits
            mode: Type of positive map
            shots: Number of measurement shots
        """
        self.n_qubits = n_qubits
        self.target_state_prep_fn = target_state_prep_fn
        self.mode = mode
        self.use_parallel = use_parallel
        self.optimizer = optimizer
        self.shots = shots
        
        # Create device
        self.dev = qml.device('default.qubit', wires=n_qubits, shots=shots)
        
        # Store Pauli strings and coefficients
        self.PauliString = {}
        self.test_state_circuit = BrickWallCircuit(n_qubits=n_qubits, **test_state_circuit_config)
        
        # Initialize Pauli decomposition
        self._setup_pauli_decomposition()
        
    def _setup_pauli_decomposition(self):
        """Setup Pauli string decomposition based on the chosen map"""
        
        if self.mode == 'partial_transpose':
            self._generate_partial_transpose_decomposition()
            
        elif self.mode == 'reduction':
            self._generate_reduction_decomposition()
    
    def _generate_partial_transpose_decomposition(self):
        """
        Generate Pauli decomposition for partial transpose on subsystem B
        T_B = ⊗_i (I + X_i - Y_i + Z_i)/2
        """
        single_qubit_coeffs = {
            'I': 1,
            'X': 1,
            'Y': -1,  # Negative!
            'Z': 1
        }
        
        pauli_ops = ['I', 'X', 'Y', 'Z']
        
        for pauli_combo_B in itertools.product(pauli_ops, repeat=int(self.n_qubits/2)):
            coeff = 1.0 / (2**int(self.n_qubits/2))
            for pauli in pauli_combo_B:
                coeff *= single_qubit_coeffs[pauli]
            
            pauli_string_A = 'I' * int(self.n_qubits/2)
            pauli_string_B = ''.join(pauli_combo_B)
            full_pauli_string = pauli_string_A + pauli_string_B
            
            self.PauliString[full_pauli_string] = coeff
    
    def _generate_reduction_decomposition(self):
        """
        Generate Pauli decomposition for reduction map on subsystem B
        R_B = (-I + X + Y + Z)/2 for each qubit
        """
        pauli_ops = ['I', 'X', 'Y', 'Z']
        
        for pauli_combo_B in itertools.product(pauli_ops, repeat=int(self.n_qubits/2)):
            coeff = 1.0 / (2**int(self.n_qubits/2))
            
            pauli_string_A = 'I' * int(self.n_qubits/2)
            pauli_string_B = ''.join(pauli_combo_B)
            full_pauli_string = pauli_string_A + pauli_string_B
            
            self.PauliString[full_pauli_string] = coeff

        self.PauliString['I'*self.n_qubits] = 1/2**int(self.n_qubits/2) - 1
    
    def _apply_pauli_string(self, pauli_string):
        """Apply Pauli operations based on string"""
        for qubit_idx, pauli_op in enumerate(pauli_string):
            if pauli_op == 'X':
                qml.PauliX(wires=qubit_idx)
            elif pauli_op == 'Y':
                qml.PauliY(wires=qubit_idx)
            elif pauli_op == 'Z':
                qml.PauliZ(wires=qubit_idx)
    
    def compute_overlap(self, pauli_string, test_state_circuit):
        """
        Compute <ψ|O(ρ)|ψ> for a given Pauli string
        
        Uses the circuit: U†(α) · O · ρ · O† · U(α) |00...0>
        """
        @qml.qnode(self.dev)
        def overlap_circuit():
            # 1. Prepare target state ρ
            self.target_state_prep_fn()
            
            # 2. Apply Pauli operation O
            self._apply_pauli_string(pauli_string)
            
            # 3. Apply test state circuit
            test_state_circuit.apply_circuit()
            
            # Return probabilities
            return qml.probs(wires=range(self.n_qubits))
        
        # Get probability of |00...0>
        probs = overlap_circuit()
        overlap = probs[0]  # First element is |00...0>
        
        return overlap
    
    def loss_function(self, test_state_circuit=None):
        """
        Compute loss L(α) = Σ r_O <ψ|O(ρ)|ψ>
        
        Returns:
            Loss value (negative indicates entanglement)
        """
        loss = 0.0
        if test_state_circuit is None:
            test_state_circuit = self.test_state_circuit
        
        for pauli_string, coeff in self.PauliString.items():
            overlap = self.compute_overlap(pauli_string, test_state_circuit)
            loss += coeff * overlap
            
        return loss

    def compute_grad(self):
        """
        Improved gradient calculation using parameter shift rule
        """
        para = self.test_state_circuit.para
        grad_set = {key: np.zeros_like(value) for key, value in para.items()}
        
        def create_circuit(modified_para):
            config = {
                "n_qubits": self.test_state_circuit.n_qubits,
                "num_layers": self.test_state_circuit.num_layers,
                "gate_type": self.test_state_circuit.gate_type,
                "para": modified_para
            }
            return BrickWallCircuit(**config)
        
        # Flatten all parameters for easier iteration
        param_list = []
        for key, value in para.items():
            indices = np.ndindex(value.shape)
            for idx in indices:
                param_list.append((key, idx))
        
        if self.use_parallel:
            grad_results = self._parallel_grad_computation(para, param_list, create_circuit)
            
            for (key, idx), grad_val in grad_results:
                grad_set[key][idx] = grad_val
        else:
            for key, idx in param_list:
                if np.random.randn(1) > 0.8:
                    grad_set[key][idx] = 0
                    continue
                grad_val = self._compute_single_gradient(para, key, idx, create_circuit)
                grad_set[key][idx] = grad_val
        
        return grad_set
    
    def _compute_single_gradient(self, para, key, idx, create_circuit):
        """
        Compute gradient for a single parameter using parameter shift rule
        """
        para_plus = self._deep_copy_para(para)
        para_minus = self._deep_copy_para(para)
        
        # Apply parameter shift (π/2 for parameter shift rule)
        para_plus[key][idx] += np.pi/2
        para_minus[key][idx] -= np.pi/2
        
        # Create circuits
        qc_plus = create_circuit(para_plus)
        qc_minus = create_circuit(para_minus)
        
        # Compute gradient
        gradient = (self.loss_function(qc_plus) - self.loss_function(qc_minus)) / 2
        
        return gradient
    
    def _parallel_grad_computation(self, para, param_list, create_circuit):
        """
        Compute gradients in parallel
        """
        def compute_grad_task(param_info):
            key, idx = param_info
            return (param_info, self._compute_single_gradient(para, key, idx, create_circuit))
        
        with ProcessPoolExecutor(max_workers=4) as executor:
            results = list(executor.map(compute_grad_task, param_list))
        
        return results
    
    def _deep_copy_para(self, para):
        """
        Efficiently deep copy parameter dictionary
        """
        return {key: value.copy() for key, value in para.items()}

    def train(self, num_epoch=100):
        log = []
        para = self.test_state_circuit.para

        for epoch in range(num_epoch):
            loss = self.loss_function()
            grad = self.compute_grad()
            log.append(loss)
            print(f"[Epoch {epoch}] Loss {loss:.4f} ")
            self.optimizer.update(w=para, grad=grad)
        
        return log

In [5]:
class EntanglementQuantification:
    def __init__(self, target_state_prep_fn, 
                test_state_circuit_config, 
                optimizer, 
                n_qubits=2, 
                depolarizing_prop=0.1,
                use_parallel=False,
                shots=10000):
        """
        Initialize for logarithmic negativity calculation
        Need ancilla qubit for variational trace norm estimation
        """
        self.n_qubits = n_qubits
        self.target_state_prep_fn = target_state_prep_fn
        self.use_parallel = use_parallel
        self.optimizer = optimizer
        self.shots = shots
        self.depolarizing_prop = depolarizing_prop

        # Need ancilla for trace norm estimation
        self.dev = qml.device('default.mixed', wires=n_qubits+1, shots=shots)
        
        # Variational circuit acts on system + ancilla
        self.test_state_circuit = BrickWallCircuit(
            n_qubits=n_qubits+1,  # System + ancilla
            **test_state_circuit_config
        )
        
        self.PauliString = {}
        self._generate_partial_transpose_decomposition()
        
    def _generate_partial_transpose_decomposition(self):
        """
        Generate Pauli decomposition for partial transpose on subsystem B
        T_B = ⊗_i (I + X_i - Y_i + Z_i)/2
        """
        single_qubit_coeffs = {
            'I': 1,
            'X': 1,
            'Y': -1,  # Negative!
            'Z': 1
        }
        
        pauli_ops = ['I', 'X', 'Y', 'Z']
        
        for pauli_combo_B in itertools.product(pauli_ops, repeat=int(self.n_qubits/2)):
            coeff = 1.0 / (2**int(self.n_qubits/2))
            for pauli in pauli_combo_B:
                coeff *= single_qubit_coeffs[pauli]
            
            pauli_string_A = 'I' * int(self.n_qubits/2)
            pauli_string_B = ''.join(pauli_combo_B)
            full_pauli_string = pauli_string_A + pauli_string_B
            
            self.PauliString[full_pauli_string] = coeff

    def loss_function(self, test_state_circuit=None):
        """
        Compute the variational trace norm estimation
        Based on: ||A||_1 = 2 max_U Tr[|0⟩⟨0|_R Q_R] - 1
        where Q_R = Tr_{AB}[U(ρ^{T_B} ⊗ |0⟩⟨0|_R)U†]
        
        For logarithmic negativity, we want to MAXIMIZE this,
        so we return negative for minimization
        """
        if test_state_circuit is None:
            test_state_circuit = self.test_state_circuit
        
        # Initialize accumulator for the variational formula
        total = 0.0
        
        # We need to apply U(ρ^{T_B} ⊗ |0⟩⟨0|)U† 
        # The partial transpose is decomposed into Pauli terms
        for pauli_string, coeff in self.PauliString.items():
            overlap = self._compute_trace_norm_overlap(pauli_string, test_state_circuit)
            total += coeff * overlap
        
        # The formula: 2 * max_U Tr[|0⟩⟨0|_R Q_R] - 1
        # We're maximizing, so return negative for optimizer
        return -total  # Negative because we minimize loss but want to maximize trace norm
    
    def _apply_pauli_string(self, pauli_string):
        """Apply Pauli operations based on string"""
        for qubit_idx, pauli_op in enumerate(pauli_string):
            if pauli_op == 'X':
                qml.PauliX(wires=qubit_idx)
            elif pauli_op == 'Y':
                qml.PauliY(wires=qubit_idx)
            elif pauli_op == 'Z':
                qml.PauliZ(wires=qubit_idx)

    def _compute_trace_norm_overlap(self, pauli_string, test_state_circuit):
        """
        Compute overlap for trace norm estimation with ancilla
        This implements the circuit from Algorithm 4 in the paper
        """
        @qml.qnode(self.dev)
        def trace_norm_circuit():
            # 1. Initialize ancilla in |0⟩ (automatic)
            
            # 2. Prepare target state on main system
            self.target_state_prep_fn(self.depolarizing_prop)
            
            # 3. Apply Pauli operation (partial transpose component)
            self._apply_pauli_string(pauli_string)
            
            # 4. Apply variational unitary U_{ABR} over system + ancilla
            test_state_circuit.apply_circuit()
            

            # 5. Measure ancilla in computational basis
            # We want probability of ancilla being in |0⟩
            ops = [qml.Identity(i) for i in range(self.n_qubits)]
            ops.append(0.5*qml.Identity(self.n_qubits) + 0.5*qml.PauliZ(self.n_qubits))
            O = ops[0]
            for i in range(1, len(ops)):
                O = O @ ops[i]

            return qml.expval(O)  # Measure ancilla (last wire)
        
        probs = trace_norm_circuit()
        # Return probability of ancilla being |0⟩
        return probs
    
    def train(self, num_epoch=100):
        """
        Train to find maximum of trace norm
        """
        log = []
        para = self.test_state_circuit.para
        
        for epoch in range(num_epoch):
            loss = self.loss_function()  # This is negative of what we want to maximize
            grad = self.compute_grad()
            
            # Convert back to positive for display (actual trace norm estimate)
            trace_norm_estimate = -loss
            
            # Logarithmic negativity
            log_negativity = np.log2(max(2*trace_norm_estimate-1, 1.0))
            
            log.append(trace_norm_estimate)
            
            print(f"[Epoch {epoch}] Trace norm: {trace_norm_estimate:.4f}, "
                  f"Log negativity: {log_negativity:.4f}")
            
            self.optimizer.update(w=para, grad=grad)
        
        return log
    
    def calculate_logarithmic_negativity(self):
        """
        After training, compute the final logarithmic negativity
        """
        # Get optimized trace norm
        trace_norm = -self.loss_function()  # Negative because we minimized negative
        
        # Apply the formula from the paper
        # ||ρ^{T_B}||_1 = 2 * optimal_value - 1
        actual_trace_norm = 2 * trace_norm - 1
        
        # Logarithmic negativity
        log_negativity = np.log2(max(actual_trace_norm, 1.0))
        
        return {
            'logarithmic_negativity': log_negativity,
            'trace_norm': actual_trace_norm,
            'raw_optimization_value': trace_norm
        }
    
    def compute_grad(self):
        """
        Improved gradient calculation using parameter shift rule
        """
        para = self.test_state_circuit.para
        grad_set = {key: np.zeros_like(value) for key, value in para.items()}
        
        def create_circuit(modified_para):
            config = {
                "n_qubits": self.test_state_circuit.n_qubits,
                "num_layers": self.test_state_circuit.num_layers,
                "gate_type": self.test_state_circuit.gate_type,
                "para": modified_para
            }
            return BrickWallCircuit(**config)
        
        # Flatten all parameters for easier iteration
        param_list = []
        for key, value in para.items():
            indices = np.ndindex(value.shape)
            for idx in indices:
                param_list.append((key, idx))
        
        if self.use_parallel:
            grad_results = self._parallel_grad_computation(para, param_list, create_circuit)
            
            for (key, idx), grad_val in grad_results:
                grad_set[key][idx] = grad_val
        else:
            for key, idx in param_list:
                if np.random.randn(1) > 0.8:
                    grad_set[key][idx] = 0
                    continue
                grad_val = self._compute_single_gradient(para, key, idx, create_circuit)
                grad_set[key][idx] = grad_val
        
        return grad_set
    
    def _compute_single_gradient(self, para, key, idx, create_circuit):
        """
        Compute gradient for a single parameter using parameter shift rule
        """
        para_plus = self._deep_copy_para(para)
        para_minus = self._deep_copy_para(para)
        
        # Apply parameter shift (π/2 for parameter shift rule)
        para_plus[key][idx] += np.pi/2
        para_minus[key][idx] -= np.pi/2
        
        # Create circuits
        qc_plus = create_circuit(para_plus)
        qc_minus = create_circuit(para_minus)
        
        # Compute gradient
        gradient = (self.loss_function(qc_plus) - self.loss_function(qc_minus)) / 2
        
        return gradient
    
    def _parallel_grad_computation(self, para, param_list, create_circuit):
        """
        Compute gradients in parallel
        """
        def compute_grad_task(param_info):
            key, idx = param_info
            return (param_info, self._compute_single_gradient(para, key, idx, create_circuit))
        
        with ProcessPoolExecutor(max_workers=4) as executor:
            results = list(executor.map(compute_grad_task, param_list))
        
        return results
    
    def _deep_copy_para(self, para):
        """
        Efficiently deep copy parameter dictionary
        """
        return {key: value.copy() for key, value in para.items()}



## Quick Example: PPT in Bell state

In [ ]:
# Define target state preparation function (Bell state)
def prepare_bell_state():
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0, 1])

# Visualize the circuit
dev_vis = qml.device('default.qubit', wires=2)

@qml.qnode(dev_vis)
def bell_circuit():
    prepare_bell_state()
    return qml.state()

# Draw the circuit
print(qml.draw(bell_circuit)())
# Or use drawer for matplotlib figure
fig, ax = qml.draw_mpl(bell_circuit)()
plt.show()

In [ ]:
test_state_circuit_config = {
    "num_layers": 1, 
    "num_RZZ": 1, 
    "gate_type": None, 
    "para": None
}

In [ ]:
optimizer = SGD(lr=5e-1)
test = EntanglementDetection(
    target_state_prep_fn=prepare_bell_state, 
    test_state_circuit_config=test_state_circuit_config, 
    optimizer=optimizer, 
    n_qubits=2,
    shots=10000
)



In [ ]:
print("Initial loss:", test.loss_function())
log = test.train(num_epoch=50)

In [ ]:
# Plot the training results
plt.figure(figsize=(10, 6))
plt.plot(log)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Entanglement Detection Training')
plt.axhline(y=0, color='r', linestyle='--', label='Separability threshold')
plt.legend()
plt.grid(True)
plt.show()

## Fig5 Reproduction

In [ ]:
class EntanglementDetection:
    def __init__(self, target_state_prep_fn, 
                test_state_circuit_config, 
                optimizer, 
                n_qubits=2, 
                use_parallel=False,
                shots=10000,
                ):
        """
        Initialize VED for entanglement detection
        
        Args:
            target_state_prep_fn: Function that prepares the target state
            n_qubits: Number of qubits
            mode: Type of positive map
            shots: Number of measurement shots
        """
        self.n_qubits = n_qubits
        self.target_state_prep_fn = target_state_prep_fn
        self.use_parallel = use_parallel
        self.optimizer = optimizer
        self.shots = shots
        
        # Create device
        self.dev = qml.device('default.mixed', wires=n_qubits+1, shots=shots)
        
        # Store Pauli strings and coefficients
        self.PauliString = {}
        self.test_state_circuit = BrickWallCircuit(n_qubits=n_qubits+1, **test_state_circuit_config)
        
        self._generate_partial_transpose_decomposition()
            
    def _generate_partial_transpose_decomposition(self):
        """
        Generate Pauli decomposition for partial transpose on subsystem B
        T_B = ⊗_i (I + X_i - Y_i + Z_i)/2
        """
        single_qubit_coeffs = {
            'I': 1,
            'X': 1,
            'Y': -1,  # Negative!
            'Z': 1
        }
        
        pauli_ops = ['I', 'X', 'Y', 'Z']
        
        for pauli_combo_B in itertools.product(pauli_ops, repeat=int(self.n_qubits/2)):
            coeff = 1.0 / (2**int(self.n_qubits/2))
            for pauli in pauli_combo_B:
                coeff *= single_qubit_coeffs[pauli]
            
            pauli_string_A = 'I' * int(self.n_qubits/2)
            pauli_string_B = ''.join(pauli_combo_B)
            full_pauli_string = pauli_string_A + pauli_string_B
            
            self.PauliString[full_pauli_string] = coeff
    
    def _apply_pauli_string(self, pauli_string):
        """Apply Pauli operations based on string"""
        for qubit_idx, pauli_op in enumerate(pauli_string):
            if pauli_op == 'X':
                qml.PauliX(wires=qubit_idx)
            elif pauli_op == 'Y':
                qml.PauliY(wires=qubit_idx)
            elif pauli_op == 'Z':
                qml.PauliZ(wires=qubit_idx)
    
    def compute_overlap(self, pauli_string, test_state_circuit):
        """
        Compute <ψ|O(ρ)|ψ> for a given Pauli string
        
        Uses the circuit: U†(α) · O · ρ · O† · U(α) |00...0>
        """
        @qml.qnode(self.dev)
        def overlap_circuit():
            # 1. Prepare target state ρ
            self.target_state_prep_fn()
            
            # 2. Apply Pauli operation O
            self._apply_pauli_string(pauli_string)
            
            # 3. Apply test state circuit
            test_state_circuit.apply_circuit()
            
            # Return probabilities
            return qml.probs(wires=range(self.n_qubits))
        
        # Get probability of |00...0>
        probs = overlap_circuit()
        overlap = probs[0]  # First element is |00...0>
        
        return overlap
    
    def loss_function(self, test_state_circuit=None):
        """
        Compute loss L(α) = Σ r_O <ψ|O(ρ)|ψ>
        
        Returns:
            Loss value (negative indicates entanglement)
        """
        loss = 0.0
        if test_state_circuit is None:
            test_state_circuit = self.test_state_circuit
        
        for pauli_string, coeff in self.PauliString.items():
            overlap = self.compute_overlap(pauli_string, test_state_circuit)
            loss += coeff * overlap
            
        return loss

    def compute_grad(self):
        """
        Improved gradient calculation using parameter shift rule
        """
        para = self.test_state_circuit.para
        grad_set = {key: np.zeros_like(value) for key, value in para.items()}
        
        def create_circuit(modified_para):
            config = {
                "n_qubits": self.test_state_circuit.n_qubits,
                "num_layers": self.test_state_circuit.num_layers,
                "gate_type": self.test_state_circuit.gate_type,
                "para": modified_para
            }
            return BrickWallCircuit(**config)
        
        # Flatten all parameters for easier iteration
        param_list = []
        for key, value in para.items():
            indices = np.ndindex(value.shape)
            for idx in indices:
                param_list.append((key, idx))
        
        if self.use_parallel:
            grad_results = self._parallel_grad_computation(para, param_list, create_circuit)
            
            for (key, idx), grad_val in grad_results:
                grad_set[key][idx] = grad_val
        else:
            for key, idx in param_list:
                if np.random.randn(1) > 0.8:
                    grad_set[key][idx] = 0
                    continue
                grad_val = self._compute_single_gradient(para, key, idx, create_circuit)
                grad_set[key][idx] = grad_val
        
        return grad_set
    
    def _compute_single_gradient(self, para, key, idx, create_circuit):
        """
        Compute gradient for a single parameter using parameter shift rule
        """
        para_plus = self._deep_copy_para(para)
        para_minus = self._deep_copy_para(para)
        
        # Apply parameter shift (π/2 for parameter shift rule)
        para_plus[key][idx] += np.pi/2
        para_minus[key][idx] -= np.pi/2
        
        # Create circuits
        qc_plus = create_circuit(para_plus)
        qc_minus = create_circuit(para_minus)
        
        # Compute gradient
        gradient = (self.loss_function(qc_plus) - self.loss_function(qc_minus)) / 2
        
        return gradient
    
    def _parallel_grad_computation(self, para, param_list, create_circuit):
        """
        Compute gradients in parallel
        """
        def compute_grad_task(param_info):
            key, idx = param_info
            return (param_info, self._compute_single_gradient(para, key, idx, create_circuit))
        
        with ProcessPoolExecutor(max_workers=4) as executor:
            results = list(executor.map(compute_grad_task, param_list))
        
        return results
    
    def _deep_copy_para(self, para):
        """
        Efficiently deep copy parameter dictionary
        """
        return {key: value.copy() for key, value in para.items()}

    def train(self, num_epoch=100):
        log = []
        para = self.test_state_circuit.para

        for epoch in range(num_epoch):
            loss = self.loss_function()
            grad = self.compute_grad()
            log.append(loss)
            print(f"[Epoch {epoch}] Loss {loss:.4f} ")
            self.optimizer.update(w=para, grad=grad)
        
        return log

## Fig6 Reproduction

In [ ]:
# Another example with Bell state
@qml.qnode(dev_vis)
def target_circuit():
    prepare_bell_state()
    return qml.state()

fig, ax = qml.draw_mpl(target_circuit)()
plt.show()

## Fig 7 Reproduction

## Fig 8 Reproduction

In [6]:
dev_mix = qml.device('default.qubit', wires=2)
def depolarizing_circuit(p=0.1):
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0, 1])
    # qml.DepolarizingChannel(p, wires=0)
    # qml.DepolarizingChannel(p, wires=1)


In [10]:
test_state_circuit_config = {
    "num_layers": 6, 
    "num_RZZ": 2, 
    "gate_type": None, 
    "para": None
}

quantifier = EntanglementQuantification(
    target_state_prep_fn=depolarizing_circuit,
    test_state_circuit_config=test_state_circuit_config,
    optimizer=SGD(lr=0.5),
    n_qubits=2,
    depolarizing_prop = 0.0,
    shots=10000
)


In [ ]:
log_history = quantifier.train(num_epoch=300)

[Epoch 0] Trace norm: 0.3049, Log negativity: 0.0000
[Epoch 1] Trace norm: 0.9699, Log negativity: 0.0000
[Epoch 2] Trace norm: 0.9828, Log negativity: 0.0000
[Epoch 3] Trace norm: 0.9851, Log negativity: 0.0000
[Epoch 4] Trace norm: 0.9861, Log negativity: 0.0000
[Epoch 5] Trace norm: 0.9875, Log negativity: 0.0000
[Epoch 6] Trace norm: 0.9925, Log negativity: 0.0000
[Epoch 7] Trace norm: 0.9892, Log negativity: 0.0000
[Epoch 8] Trace norm: 0.9940, Log negativity: 0.0000
[Epoch 9] Trace norm: 0.9961, Log negativity: 0.0000
[Epoch 10] Trace norm: 0.9996, Log negativity: 0.0000
[Epoch 11] Trace norm: 1.0074, Log negativity: 0.0212
[Epoch 12] Trace norm: 1.0005, Log negativity: 0.0013
[Epoch 13] Trace norm: 1.0011, Log negativity: 0.0032
[Epoch 14] Trace norm: 1.0086, Log negativity: 0.0246
[Epoch 15] Trace norm: 1.0079, Log negativity: 0.0228
[Epoch 16] Trace norm: 1.0080, Log negativity: 0.0228
[Epoch 17] Trace norm: 1.0085, Log negativity: 0.0242
[Epoch 18] Trace norm: 1.0100, Log neg

KeyboardInterrupt: 